In [ ]:
import paramiko
from scp import SCPClient
import os
import sys
import json

# ================= VARIABLES DE ENTRADA  =================
MODEL_NAME = "yolov11n_accuracy"

# 1. RUTA DEL MODELO HEF LOCAL
LOCAL_HEF_PATH = f"models/binaries/{MODEL_NAME}.hef"

# 2. ¿EJECUTAR TODO?
SKIP_RPI = False

# 3. DATOS DE LA RASPBERRY PI
RPI_CONFIG = {
    'ip': '192.168.1.xxx',
    'username': 'USERNNAME',
    'password': '',
    'work_dir': '/home/USERNAME/Desktop/hailo/runtime'
}

# 4. DATOS DE LA MÁQUINA DE ANÁLISIS (REMOTE)
REMOTE_MACHINE = {
    'ip': 'LINUX_MACHINE_IP',
    'username': 'USERNAME',
    'password': '',
    'work_dir': '/home/USERNAME/hailo/data_out/results',
    'models_har_dir': '/home/USERNAME/hailo/models/intermediate',
    'models_hef_dir': '/home/USERNAME/hailo/models/binaries',
    'venv_activate_cmd': 'source /home/USERNAME/hailo/.venv/bin/activate'
}

# 5. CONFIGURACIÓN DE RESULTADOS LOCALES
LOCAL_RESULTS_BASE_DIR = "data_out"
EXPERIMENT_FOLDER_NAME = f"{MODEL_NAME}"

# Nombre del archivo temporal
TEMP_JSON_NAME = f"runtime_data_{MODEL_NAME}.json"

# ================= LÓGICA DEL SCRIPT =================

def create_ssh_client(config):
    """Crea una conexión SSH con timeout y sin buscar llaves locales"""
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    try:
        ssh.connect(
            config['ip'],
            username=config['username'],
            password=config['password'],
            look_for_keys=False,
            timeout=10
        )
        return ssh
    except Exception as e:
        print(f"❌ Error conectando a {config['ip']}: {e}")
        return None

def step_1_raspberry_execution(local_hef):
    print(f"\n--- PASO 1: Ejecución en Raspberry Pi ({RPI_CONFIG['ip']}) ---")
    filename_hef = os.path.basename(local_hef)
    ssh = create_ssh_client(RPI_CONFIG)
    if not ssh: return False

    try:
        with SCPClient(ssh.get_transport()) as scp:
            ssh.exec_command(f"mkdir -p {RPI_CONFIG['work_dir']}")
            remote_hef_path = f"{RPI_CONFIG['work_dir']}/{filename_hef}"
            remote_json_path = f"{RPI_CONFIG['work_dir']}/{TEMP_JSON_NAME}"

            print(f"Subiendo {filename_hef}...")
            scp.put(local_hef, remote_hef_path)

            cmd = f"hailortcli run2 -m raw measure-fw-actions --output-path {remote_json_path} set-net {remote_hef_path}"
            print(f"Ejecutando inferencia en Hardware...")

            stdin, stdout, stderr = ssh.exec_command(cmd)
            status = stdout.channel.recv_exit_status()

            print(f" Recuperando salida...")
            scp.get(remote_json_path, TEMP_JSON_NAME)

            try:
                with open(TEMP_JSON_NAME, 'r') as f:
                    content = f.read()
                    if not content.strip():
                        print("ERROR CRÍTICO: El archivo recibido está VACÍO.")
                        return False
                    json.loads(content)
                    print("JSON válido recibido.")
                    return True
            except json.JSONDecodeError:
                print("\nERROR EN LA RASPBERRY PI DETECTADO:")
                print(content)
                return False
    except Exception as e:
        print(f"Excepción en Paso 1: {e}")
        return False
    finally:
        ssh.close()

def step_2_remote_analysis(local_hef):
    print(f"\n--- PASO 2: Envío a Máquina de Análisis ({REMOTE_MACHINE['ip']}) ---")

    filename_hef = os.path.basename(local_hef)
    model_base_name = os.path.splitext(filename_hef)[0]
    remote_har_name = f"{model_base_name}_quantized.har"
    remote_hef_name = f"{model_base_name}.hef"

    ssh = create_ssh_client(REMOTE_MACHINE)
    if not ssh: return False

    try:
        with SCPClient(ssh.get_transport()) as scp:
            ssh.exec_command(f"mkdir -p {REMOTE_MACHINE['work_dir']}")
            remote_json_dest = f"{REMOTE_MACHINE['work_dir']}/{TEMP_JSON_NAME}"

            print(f"✈️ Enviando JSON...")
            scp.put(TEMP_JSON_NAME, remote_json_dest)

            out_html = f"{REMOTE_MACHINE['work_dir']}/{model_base_name}.html"
            out_csv = f"{REMOTE_MACHINE['work_dir']}/{model_base_name}.csv"
            path_to_har = f"{REMOTE_MACHINE['models_har_dir']}/{remote_har_name}"
            path_to_hef = f"{REMOTE_MACHINE['models_hef_dir']}/{remote_hef_name}"

            profiler_cmd = (
                f"hailo profiler {path_to_har} "
                f"--hef {path_to_hef} "
                f"--runtime-data {remote_json_dest} "
                f"--out-path {out_html} "
                f"--csv {out_csv}"
            )

            full_cmd = f"{REMOTE_MACHINE['venv_activate_cmd']} && {profiler_cmd}"

            print(f"Ejecutando Profiler Remoto...")
            stdin, stdout, stderr = ssh.exec_command(full_cmd)

            exit_status = stdout.channel.recv_exit_status()

            if exit_status == 0:
                print("\n¡PROFILING EXITOSO!")

                # --- LÓGICA DE DESCARGA A CARPETA PERSONALIZADA ---
                target_dir = os.path.join(LOCAL_RESULTS_BASE_DIR, EXPERIMENT_FOLDER_NAME)

                # Crear la carpeta local si no existe
                if not os.path.exists(target_dir):
                    os.makedirs(target_dir)
                    print(f"Carpeta creada: {target_dir}")

                print(f" Descargando resultados a: {target_dir}...")

                # Descargamos los archivos generados
                scp.get(out_html, os.path.join(target_dir, f"{model_base_name}.html"))
                scp.get(out_csv, os.path.join(target_dir, f"{model_base_name}.csv"))

                if os.path.exists(TEMP_JSON_NAME):
                    os.replace(TEMP_JSON_NAME, os.path.join(target_dir, TEMP_JSON_NAME))

                print(f"✨ Proceso finalizado. Archivos guardados en '{EXPERIMENT_FOLDER_NAME}'.")
            else:
                error = stderr.read().decode()
                print("\nFallo en el profiler remoto:")
                print(error)

            return True

    except Exception as e:
        print(f"Excepción en Paso 2: {e}")
        return False
    finally:
        ssh.close()

# --- EJECUCIÓN ---
if os.path.exists(LOCAL_HEF_PATH):
    success = True
    if not SKIP_RPI:
        success = step_1_raspberry_execution(LOCAL_HEF_PATH)

    if success:
        step_2_remote_analysis(LOCAL_HEF_PATH)
else:
    print(f" No encuentro el archivo HEF en: {LOCAL_HEF_PATH}")

In [ ]:
import pandas as pd
import json
import io
import os

# ==========================================
# CONFIGURACIÓN
# ==========================================
CSV_FILE = "data_out\\yolov11n_balanced\\yolov11n_balanced.csv"
JSON_FILE = "data_out\\yolov11n_balanced\\runtime_data_yolov11n_balanced.json"

class HailoProfilerAnalyzer:
    def __init__(self, csv_path, json_path):
        self.csv_path = csv_path
        self.json_path = json_path
        self.df_global = None
        self.df_contexts = None
        self.df_layers = None

    def parse_hailo_multi_csv(self):
        """Separa el CSV anómalo de Hailo en 3 DataFrames distintos."""
        if not os.path.exists(self.csv_path):
            raise FileNotFoundError(f"No se encuentra el archivo: {self.csv_path}")

        with open(self.csv_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        global_lines, context_lines, layer_lines = [], [], []
        current_block = None

        for line in lines:
            line_strip = line.strip()
            if not line_strip: continue

            # Máquina de estados para detectar en qué bloque estamos
            if line_strip.startswith('model_name'): current_block = 'global'
            elif line_strip.startswith('context_name'): current_block = 'context'
            elif line_strip.startswith('layer_name'): current_block = 'layer'

            # Asignar la línea al bloque correspondiente
            if current_block == 'global': global_lines.append(line_strip)
            elif current_block == 'context': context_lines.append(line_strip)
            elif current_block == 'layer': layer_lines.append(line_strip)

        # Convertir a DataFrames de Pandas
        if global_lines:
            self.df_global = pd.read_csv(io.StringIO('\n'.join(global_lines)))
        if context_lines:
            self.df_contexts = pd.read_csv(io.StringIO('\n'.join(context_lines)))
        if layer_lines:
            self.df_layers = pd.read_csv(io.StringIO('\n'.join(layer_lines)))

    def generate_thesis_report(self):
        """Genera el reporte de texto formateado para la memoria del TFM."""
        print("\n" + "="*70)
        print(" REPORTE TÉCNICO DE TELEMETRÍA HARDWARE (HAILO-8L)")
        print("="*70)

        # 1. ANÁLISIS GLOBAL
        if self.df_global is not None:
            print("\n[1] RESUMEN GLOBAL DEL ACELERADOR")
            print("-" * 70)
            row = self.df_global.iloc[0]
            print(f"Modelo:                  {row.get('model_name', 'N/A')}")
            print(f"Operaciones (MACS):      {float(row.get('macs_per_image', 0)) / 1e9:.2f} GMACs")
            print(f"Throughput Entrada:      {row.get('gross_input_throughput', 'N/A')}")
            print(f"Throughput Salida:       {row.get('gross_output_throughput', 'N/A')}")
            print(f"Nº de Contextos (SRAM):  {row.get('number_of_contexts', 'N/A')}")

        # 2. ANÁLISIS DE BUS PCIE (CONTEXTOS)
        if self.df_contexts is not None:
            print("\n[2] ANÁLISIS DE CUELLO DE BOTELLA: BUS PCIE (Inter-Context Spilling)")
            print("-" * 70)

            # Filtramos la fila "contexts_total" si existe
            df_ctx = self.df_contexts[~self.df_contexts['context_name'].str.contains('total', case=False, na=False)]

            total_spill_mb = 0
            for _, ctx in df_ctx.iterrows():
                name = ctx.get('context_name', 'Unknown')
                out_bytes = float(ctx.get('inter_context_out_bytes', 0)) / (1024*1024)
                in_bytes = float(ctx.get('inter_context_in_bytes', 0)) / (1024*1024)
                total_spill_mb += (out_bytes + in_bytes)

                if out_bytes > 0 or in_bytes > 0:
                    print(f" - {name}:")
                    if out_bytes > 0: print(f"     -> Expulsado a RAM (DDR): {out_bytes:.2f} MB")
                    if in_bytes > 0:  print(f"     <- Leído desde RAM (DDR): {in_bytes:.2f} MB")

            print(f"\n   >> DIAGNÓSTICO: El chip está transfiriendo {total_spill_mb:.2f} MB por frame ")
            print("   >> a través del bus PCIe debido a la fragmentación de contextos.")
            print("   >> SOLUCIÓN SUGERIDA: 'compression_level=1' y 'max_contexts=2'.")

        # 3. ANÁLISIS DE CAPAS (UTILIZACIÓN DE SILICIO)
        if self.df_layers is not None:
            print("\n[3] ANÁLISIS DE EFICIENCIA DE CÓMPUTO EN SILICIO (Top 5 Ineficientes)")
            print("-" * 70)

            # Buscar capas con baja utilización de MACs (Multiplicador-Acumulador)
            if 'mac_computation_utilization' in self.df_layers.columns:
                # Limpiar datos no numéricos si los hay y ordenar de menor a mayor utilización
                df_layers_clean = self.df_layers.copy()
                df_layers_clean['mac_computation_utilization'] = pd.to_numeric(df_layers_clean['mac_computation_utilization'], errors='coerce')
                worst_layers = df_layers_clean.sort_values('mac_computation_utilization', ascending=True).head(5)

                for _, layer in worst_layers.iterrows():
                    name = layer['layer_name']
                    util = layer['mac_computation_utilization'] * 100 # a porcentaje
                    ctx = layer.get('context', 'N/A')
                    print(f" - Capa: {name:<20} | Contexto: {ctx:<18} | Uso de MACs: {util:.2f}%")
            else:
                print(" No se encontró la métrica de utilización de MACs en las capas.")

        # 4. ANÁLISIS DEL RUNTIME JSON
        print("\n[4] DATOS DE EJECUCIÓN DINÁMICA (Runtime JSON)")
        print("-" * 70)
        try:
            with open(self.json_path, 'r') as f:
                runtime_data = json.load(f)

            runs = runtime_data.get('runs', [])
            if runs:
                run = runs[0]
                fps = run.get('fps', 0)
                mean_act_ms = run.get('mean_activation_time_ms', 0)
                print(f"FPS Simulados (Hardware):  {fps:.2f} FPS")
                print(f"Tiempo de Activación SRAM: {mean_act_ms:.4f} ms por frame")
        except Exception as e:
            print(f"No se pudo extraer métricas del JSON: {e}")

        print("="*70 + "\n")

# ==========================================
# EJECUCIÓN
# ==========================================
if __name__ == "__main__":
    try:
        analyzer = HailoProfilerAnalyzer(CSV_FILE, JSON_FILE)
        analyzer.parse_hailo_multi_csv()
        analyzer.generate_thesis_report()
    except Exception as e:
        print(f"Error durante la ejecución del análisis: {e}")